In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import requests
import time
def get_smiles(cid):
    time.sleep(0.2)
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/IsomericSmiles/JSON"
    r = requests.get(url, timeout=30)
    if not r.ok:
        return None
    props = r.json().get("PropertyTable", {}).get("Properties", [])
    if not props:
        return None
    return props[0].get("SMILES")
print(get_smiles(7565))

C1=CC=C(C=C1)OC2=CC=C(C=C2)Br


In [ ]:
import requests

def get_targets_for_assay(aid):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/{aid}/targets/ProteinAccession/JSON"
    r = requests.get(url, timeout=30)
    if r.ok:
        return r.json().get("InformationList", [{}]).get('Information',[{}])[0].get('ProteinAccession')
    return []

def get_assays_for_cid(cid):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/aids/JSON"
    r = requests.get(url, timeout=30)
    if r.ok:
        return r.json().get('InformationList',[{}]).get('Information',[{}])[0].get('AID')
    return []

def get_targets_for_cid(cid):
    targets = set()
    aids = get_assays_for_cid(cid)
    for aid in aids:
        posstarget = get_targets_for_assay(aid)
        for acc in posstarget:
            targets.add(acc)
    return list(targets)

In [ ]:
import pandas as pd

chems = pd.read_csv('/content/drive/MyDrive/Work/chemicalids.csv')
chems = chems[chems['type'] == 'c']
chems['chemID'] = chems['chemID'].str[5:]
print(chems.head())



       chemID       cids type
1     C070055      35823    c
2     C024713      12389    c
3  D000077612     166548    c
4     C538809  131634859    c
5  C000626479       7565    c


In [ ]:
file_path = '/content/drive/MyDrive/Work/chemts.csv'
with open(file_path, 'x') as file:
    print(f"File '{file_path}' created successfully.")

FileExistsError: [Errno 17] File exists: '/content/drive/MyDrive/Work/chemsmiles.csv'

In [ ]:
known = pd.read_csv('/content/drive/MyDrive/Work/chemts.csv')
print(known.head())


EmptyDataError: No columns to parse from file

In [ ]:
chemicals = chems[~chems["chemID"].isin(known["chemID"])] | chems


NameError: name 'known' is not defined

In [ ]:
import csv
from tqdm import tqdm
with open('/content/drive/MyDrive/Work/chemts.csv',mode='w',newline="") as file:
    writer = csv.writer(file)
    for chem in tqdm(chems.itertuples()):
        # des = fetch_mesh_scope(chem.chemID[5:])
        try:
            targets = get_targets_for_cid(chem.cids)
            writer.writerow([chem.chemID,chem.cids,targets])
            file.flush()
        except:
            print('error')

41it [59:35, 184.62s/it]